In [ ]:
#some raw script not provided in https://github.com/MasahiroNakano-hub/ImmuNexUT_SLE

setwd("/lulabdata3/baopengfei/projects/motif-RBP-EDA/")
options(scipen = 4,stringsAsFactors = F,digits = 5)
options(bedtools.path = "/lulabdata3/baopengfei/biosoft")
library(bedtoolsr)
library(conflicted)
#unloadNamespace("conflicted")
conflict_prefer("colnames", "base")
conflict_prefer("count", "dplyr")
conflict_prefer("mutate", "dplyr")
conflict_prefer("unname", "base")

options(stringsAsFactors=F)
suppressPackageStartupMessages(library(RColorBrewer))
suppressPackageStartupMessages(library(ggsci))
suppressPackageStartupMessages(library(lme4))
suppressPackageStartupMessages(library(variancePartition))


# load all func.
source("/lulabdata3/baopengfei/projects/motif-RBP-EDA/scripts/ImmuNexUT_SLE/Code/my.source.R")
source("/lulabdata3/baopengfei/projects/WCHSU-FTC/exSeek-dev/scripts/util.R")




dst <- "TCGA_small16"
dedup <- "_all"
diffpre <- "/lulabdata3/baopengfei/projects/motif-RBP-EDA"
pre="/lulabdata3/baopengfei/projects/WCHSU-FTC"
outpre=pre #"/data2/lulab1/bpf/projects/WCHSU-FTC"
dir.create(paste0(diffpre,"/output/",dst),recursive = T,showWarnings = F) # ,"/",disease.label,"vs",normal.label


[conflicted] Will prefer base::colnames over any other package.
[conflicted] Will prefer dplyr::count over any other package.
[conflicted] Will prefer dplyr::mutate over any other package.
[conflicted] Will prefer base::unname over any other package.
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.3     ✔ tibble    3.2.1
✔ purrr     1.0.2     ✔ tidyr     1.3.1


In [ ]:
## table1: read sample meta
sample.table0 <- read.csv( paste0(pre,"/exSeek-dev/data/",dst,"/sample_table.txt"),check.names = F,sep = "\t",header = T,encoding="UTF-8")
rownames(sample.table0) <- sample.table0$sample
table(sample.table0$group)
table(sample.table0$cell)


# replace new clinica from TCGAbiolinks
tmp <- data.table::fread("/lulabdata3/baopengfei/projects/tcga/data/TCGA-all_clinical/TCGAbiolinks_clinical.tsv",data.table = F,sep = '\t',header = T,stringsAsFactors = F,check.names = F)
colnames(tmp)[1] <- "patient"
tmp <- tmp[,c("patient","project","age","gender","race","ajcc_stage","ajcc_n","prior_treatment","cigarettes_per_day","years_smoked","days_to_death")]

sample.table0$age <- NULL
sample.table0$gender <- NULL
sample.table0$race <- NULL
sample.table0 <- dplyr::left_join(sample.table0, tmp, by="patient")
#sample.table0$ajcc_stage <- NULL # use tumor_stage
#table((sample.table0$ajcc_n=="")) 
#as.numeric(tmp$age[match(patient,tmp$submitter_id)])
#hist(sample.table0$age)
#summary(sample.table0$age)
table(sample.table0$ajcc_stage)
sample.table0$tumor_stage <- sample.table0$ajcc_stage

tmp <- read.table(paste0("/lulabdata3/baopengfei/projects/tcga/data/TCGA-all_clinical/","UCEC","_clinical/","UCEC","_clinical.tsv"), quote = "",sep = "\t", header = T) # HNSC_gdc_sample_sheet.2024-01-09.tsv
#colnames(tmp)
tmp <- tmp[,c("submitter_id","figo_stage")]
colnames(tmp) <- c("patient","figo_stage")
sample.table0 <- dplyr::left_join(sample.table0, tmp, by="patient")
sample.table0$tumor_stage[sample.table0$cell %in% c("TCGA-UCEC_small")] <- sample.table0$figo_stage[sample.table0$cell %in% c("TCGA-UCEC_small")]

tmp <- read.table(paste0("/lulabdata3/baopengfei/projects/tcga/data/TCGA-all_clinical/","HNSC","_clinical/","HNSC","_clinical.tsv"), quote = "",sep = "\t", header = T) # HNSC_gdc_sample_sheet.2024-01-09.tsv
#colnames(tmp)
table(tmp$ajcc_clinical_stage)
tmp <- tmp[,c("submitter_id","ajcc_clinical_stage")]
colnames(tmp) <- c("patient","ajcc_clinical_stage")
sample.table0 <- dplyr::left_join(sample.table0, tmp, by="patient")
sample.table0$tumor_stage[sample.table0$cell %in% c("TCGA-HNSC_small")] <- sample.table0$ajcc_clinical_stage[sample.table0$cell %in% c("TCGA-HNSC_small")]

#table(sample.table0$cell,sample.table0$tumor_stage)
#table(sample.table0$figo_stage)
## TNM: https://baike.baidu.com/item/TNM%E5%88%86%E6%9C%9F%E7%B3%BB%E7%BB%9F/10700513
# NX means the nearby lymph nodes can't be assessed, for example, if they were previously removed.
# N0 means nearby lymph nodes do not contain cancer.
# N1, N2, N3: These numbers are based on the number of lymph nodes involved and how much cancer is found in them. The higher the N number, the greater the extent of the lymph node involvement.
#table(sample.table0$tumor_stage %in% c("\'--","not reported"))
#sample.table0$tumor_stage[sample.table0$tumor_stage %in% c("\'--","not reported")] <- sample.table0$ajcc_stage[sample.table0$tumor_stage %in% c("\'--","not reported")]
#sample.table0$tumor_stage[sample.table0$tumor_stage %in% c("\'--","","not reported")] <- sample.table0$ajcc_n[sample.table0$tumor_stage %in% c("\'--","","not reported")]
sample.table0$tumor_stage <- tolower(sample.table0$tumor_stage)
sample.table0$tumor_stage <- gsub("stage ia|stage ib|stage ic","stage i",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("stage iia|stage iib|stage iic","stage ii",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("stage iiic1|stage iiic2|stage iii1|stage iii2|stage iiia|stage iiib|stage iiic","stage iii",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("stage iva|stage ivb|stage ivc","stage iv",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("stage x","",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("t2a|t2b|t2c|t2d","stage ii",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("t3a|t3b|t3c|t3d","stage iii",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("t4a|t4b|t4c|t4d|t4","stage iv",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("n0","stage i",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("n1","stage i",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("n2a|n2b|n2c|n2d|n2","stage iii",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("n3a|n3b|n3c|n3d|n3","stage iii",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("n4a|n4b|n4c|n4d|n4","stage iv",sample.table0$tumor_stage, perl = T)
sample.table0$tumor_stage <- gsub("nx","",sample.table0$tumor_stage, perl = T)
sample.table0$group <- gsub("01A","Tumor",sample.table0$group)
sample.table0$group <- gsub("11A","Normal",sample.table0$group)
sample.table0$tumor_stage[sample.table0$group=="Normal"] <- "-"
table(sample.table0$tumor_stage,sample.table0$group)
##longer pattern to forward !!


## table2: read count mat
#count <- read.table( paste0(outpre,"/output/",dst,"/call_peak",dedup,"/count_matrix/cfpeakCNN_b5_d50_p1.txt"),header = T,sep = "\t",stringsAsFactors = F,check.names = F)
count0 <- data.table::fread( paste0(outpre,"/output/",dst,"/call_peak",dedup,"/count_matrix/cfpeakCNN_b5_d50_p1.txt"), header = T,sep = "\t",stringsAsFactors = F,check.names = F, data.table = F)
#count0 <- data.table::fread( paste0(outpre,"/output/",dst,"/call_peak",dedup,"/count_matrix/cfpeakCNN_b5_d50_p1_1042_rmPatient.txt"), header = T,sep = "\t",stringsAsFactors = F,check.names = F, data.table = F)
#count0 <- data.table::fread( paste0(outpre,"/output/",dst,"/call_peak",dedup,"/count_matrix/cfpeakCNN_b5_d50_p1_1041_rmDate.txt"), header = T,sep = "\t",stringsAsFactors = F,check.names = F, data.table = F)
rownames(count0) <- count0$feature
count0$feature <- NULL


## define common vec/var
cmp.df <- data.frame(row.names=c("state","tumorigenesis","metastasis","relapse"),"col"=c("group","group1","group2","group3"),"pos.grp"=c("Tumor","Primary","Yes","Relapse"),"neg.grp"=c("Normal","Normal","No","Primary"))
sample.table0$group <- gsub("01A","Tumor",sample.table0$group, perl = T)
sample.table0$group <- gsub("11A","Normal",sample.table0$group, perl = T)
sample.table0$group2 <- ""
sample.table0$group2[sample.table0$group=="Tumor" ] <- "NA"
sample.table0$group2[sample.table0$tumor_stage %in% c("stage i","stage ia","stage ib","stage ii","stage iia","stage iib") & sample.table0$group=="Tumor" ] <- "No"
sample.table0$group2[sample.table0$tumor_stage %in% c("stage iii","stage iiia","stage iiib","stage iiic","stage iv") & sample.table0$group=="Tumor" ] <- "Yes"
sample.table0$group2[ sample.table0$group=="Normal" ] <- "-"
table(sample.table0$cell,sample.table0$group2)
table(sample.table0$cell,sample.table0$tumor_stage)

sample.table0$group1 <- ""
sample.table0$group1[ sample.table0$group=="Normal" ] <- "Normal"
sample.table0$group1[ sample.table0$group=="Tumor" & sample.table0$group2 %in% c("No","NA") ] <- "Primary"
sample.table0$group1[ sample.table0$group=="Tumor" & sample.table0$group2=="Yes" ] <- "Metastasis"
table(sample.table0$cell,sample.table0$group1)


## update sample tables by latest peak matrix
sample.table0 <- sample.table0[sample.table0$sample %in% colnames(count0),]
sample.table0$cellShort <- gsub("TCGA-|_small","",sample.table0$cell,perl = T)
sample.table0$cell <- sample.table0$cellShort


cell.table <- read.table("/lulabdata3/baopengfei/projects/WCHSU-FTC/exSeek-dev/data/TCGA_small16/TCGA_superclass.txt",header = T,check.names = F,sep = "\t")
#table(sample.table0$cell %in% cell.table$Cancer)
cell.table <- cell.table[cell.table$Cancer %in% unique(sample.table0$cell),]
colnames(cell.table)[c(1,4)] <- c("cell","lineage")
cell.table$cellShort <- cell.table$cell
sample.table0$lineage <- cell.table$lineage[match(sample.table0$cell,cell.table$cell)]
unique(sample.table0$lineage )
#table(is.na(cell.table$lineage))

#cellShort <- unique(sample.table0$cellShort)
#table(sample.table0$group,sample.table0$cellShort)
colorList <- list()
# cellShort <- c("STAD","COAD","LIHC","HNSC","LUAD","PRAD","BRCA",
#                "THCA","LUSC","ESCA","BLCA","CHOL","UCEC","KIRC","KIRP","KICH")
cellShort <- c("ESCA","STAD","COAD","LIHC","CHOL",   
               "HNSC",    
               "LUAD","LUSC",
               "BLCA","KIRC","KIRP","KICH",
               "THCA",
               "UCEC",
               "PRAD","BRCA"
               )
#table(cellShort %in% cell.table$cell)
cellColor <- c(#"NC"="grey60","Normal"="grey60",
               "Pancreatic disease"="grey30","Lung disease"="grey30","Colorectal disease"="grey30","Stomach disease"="grey30","Liver disease"="grey30","Prostate disease"="grey30","Immune disease"="grey30","Other disease"="grey30",
               "CRC"="orange3","COAD"="orange3",
               "GC"="firebrick3","STAD"="firebrick3",
               "ESCA"="#EFD493FF",
               "PACA"="salmon","PAAD"="salmon","PDAC"="salmon",
               "Cholangiocarcinoma"="#DBBC92FF","CHOL"="#DBBC92FF","HCC"="#FBBC92FF","LIHC"="#FBBC92FF",
               "HNSC"="#FDF192FF",
               "THCA"="#97B792",

               "NSCLC"="#6876B1FF","LUCA"="#7876B1FF","LUAD"="#7876B1FF","LUSC"="#9876B1FF",
               "BRCA"="#EE4C97FF",
               "PRAD"="steelblue1","PRCA"="steelblue1",

               "GBM"="seagreen1",
               "LAML"="purple",
              
               "BLCA"="#91C5DE",
               "RECA"="#B8862FFF",
               "URCA"="#71853DFF","UCEC"="#7A91B8",
               "KIRC"="seagreen4","KIRP"="seagreen1","KICH"="lightgreen",
               "OVCA"="#F48194FF","ENCA"="#FB12FF",
               "Sarcoma"="#7777B0FF","Lymphoma"="#9972B9FF","Melanoma"="#764F5CFF","Myeloma"="#6A7B71FF"
               ) #pal_nejm_adaptive()(10)
colorList[["cell"]] <- cellColor
colorList[["cellShort"]] <- cellColor

#unique(cell.table$lineage)
lineageShort <- c("Digest","Head&Neck","Thoracic","Urinary","Thyroid","Reproductive","Prostate","Breast")
lineageColor <- c(
                "Digest"="orange",
                "Head&Neck"="#FDF192FF",
                "Thoracic"="#7856B1FF",
                "Urinary"="steelblue3",
                "Thyroid"="salmon",
                "Reproductive"="yellow4",
                "Prostate"="steelblue1",
                "Breast"="#EE4C97FF"
                # "Brain","Eye",
                # "Skin&SoftTissue","Hematopoietic",
                # "Pleura","AdrenalGland","Thyroid"
                )

cell.table$lineage <- factor(cell.table$lineage,levels = lineageShort)
cell.table$cellShort <- factor(cell.table$cellShort,levels = cellShort)
table(cell.table$lineage)
cell.table <- cell.table[order(cell.table$lineage,cell.table$cellShort,decreasing = F),]
colorList[["lineage"]] <- lineageColor

groupShort <- c("Normal","Tumor")
groupColor <- c("Normal"="grey60","Tumor"="firebrick2")
group1Short <- c("Normal","Primary")
group1Color <- c("Normal"="grey60","Primary"="firebrick2")
group2Short <- c("No","Yes")
group2Color <- c("No"="salmon","Yes"="firebrick4")
colorList[["group"]] <- groupColor
colorList[["group1"]] <- c(group1Color,"Metastasis"="firebrick4")
colorList[["group2"]] <- group2Color
groupAllShort <- c("Normal"="grey60","Primary"="firebrick2","Metastasis"="firebrick4","No"="salmon","Yes"="firebrick4")

#DEG <- c("Tumorigenesis only","Metastasis only","Both significant","Not significant")
DEGfactor <- c("Not significant", "Tumorigenesis only", "Metastasis only", "Both significant")
DEGColor <- c("Tumorigenesis only"="#0072B5FF","Metastasis only"="#BC3C29FF","Both significant"="#E18727FF","Not significant"="#cccccc")
colorList[["DEG"]] <- DEGColor

for (i in c("gender","race","tumor_stage")){
  colorList[[i]] <- pal_npg_adaptive()( length(unique(sample.table0[[i]])) )
  names(colorList[[i]]) <- unique(sample.table0[[i]])
}



## append ESTIMATE purity et al
tmp <- data.table::fread(file = "/lulabdata3/baopengfei/projects/tcga/ref/TCGAbiolinks_ESTIMATE.txt",sep = "\t",header = T, check.names = F, stringsAsFactors = F,data.table = F)
tmp <- tmp[,c(1,3:ncol(tmp))]
colnames(tmp)[1] <- "tissue_id"
sample.table0 <- dplyr::left_join(sample.table0,tmp,by="tissue_id")
rownames(sample.table0) <- sample.table0$sample
# table(is.na(sample.table0$CPE))
# cor.test(sample.table0$CPE,sample.table0$LUMP)
# cor.test(sample.table0$CPE,sample.table0$IHC)

## append purity metainfo
# priList <- list()
# for (i in paste0("/lulabdata3/wangtaiwei/project/RNA/SLE/output/ImmReg/pan_cancer/",cellShort,"_purity_scores.txt") ){
#   purity <- read.table(i,header = T,row.names = 1,check.names = F,stringsAsFactors = F,sep = "\t")
#   # table(duplicated(purity$sample))
#   purity$tissue_id <- substr(purity$sample,1,16)
#   priList[[i]] <- purity
# }
# priList <- as.data.frame(do.call(rbind,priList))
# data.table::fwrite(priList,paste0(pre,"/exSeek-dev/data/",dst,"/purity.txt"),quote = F, sep = '\t',row.names = F,col.names = T )
tmp <- data.table::fread(file = paste0(pre,"/exSeek-dev/data/",dst,"/purity.txt"),sep = "\t",header = T, check.names = F, stringsAsFactors = F,data.table = F)
tmp <- tmp[tmp$tissue_id %in% sample.table0$tissue_id,]
table(sample.table0$tissue_id %in% tmp$tissue_id)

tmp$sample <- NULL
table(duplicated(tmp$tissue_id))
tmp <- tmp[!duplicated(tmp$tissue_id),]
sample.table0 <- dplyr::left_join(sample.table0,tmp,by="tissue_id")
sample.table0$purity <- as.numeric(sample.table0$purity)
#table(is.na(sample.table0$purity))
#tmp2 <- sample.table0[is.na(sample.table0$purity),]
#data.table::fwrite(tmp2,paste0(pre,"/exSeek-dev/data/",dst,"/purity_NA.txt"),quote = F, sep = '\t',row.names = F,col.names = T ) # not matched mRNA-seq ?
#data.table::fwrite(x = sample.table0, paste0(pre,"/exSeek-dev/data/",dst,"/sample_table2.txt"), sep = "\t", quote = F, row.names = F, col.names = T )
rownames(sample.table0) <- sample.table0$sample
#some samples lost for NA purity
sample.table0 <- sample.table0[!is.na(sample.table0$purity),]


## get libsize
libSizePath <- paste0(outpre,"/output/",dst,"/call_peak",dedup,"/count_matrix/EM.libSize")
tmp <- read.table(libSizePath,header = F,sep = "\t",check.names = F,row.names = 1)
# tmp <- tmp[colnames(count),]
tmp <- tmp[sample.table0$sample,,drop=F]
externalLibSize <- tmp$V2 #[,"star_hg38_v38_dedup"]
#names(externalLibSize) <- colnames(count) # [2:ncol(count)]
names(externalLibSize) <- sample.table0$sample


# #rm(list="logcpm0")
# logcpm0 <- data.table::fread( paste0(outpre,"/output/",dst,"/call_peak",dedup,"/count_matrix/cfpeakCNN_b5_d50_p1_CPM_rmPurity.txt"), sep = "\t",check.names = F,header = T,data.table = F,stringsAsFactors = F )
# rownames(logcpm0) <- logcpm0$feature
# logcpm0$feature <- NULL
logcpm0 <- data.table::fread( paste0(outpre,"/output/",dst,"/call_peak",dedup,"/count_matrix/cfpeakCNN_b5_d50_p1_CPM.txt"), sep = "\t",check.names = F,header = T,data.table = F,stringsAsFactors = F )
rownames(logcpm0) <- logcpm0$gene_id
logcpm0$gene_id <- NULL
summary(logcpm0[,2])
# Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
# 0       1       1      28       1  470051 
logcpm0 <- log2(logcpm0+1)
dim(logcpm0) # 54354  3436



## filter
testID <- read.table( paste0("/lulabdata3/baopengfei/projects/WCHSU-FTC/exSeek-dev/data/",dst,"/sample_ids_test.txt"), header = F,check.names = F,stringsAsFactors = F,sep = "\t")$V1
length(testID) # 2011  2068
dim(sample.table0) # 3507  3436
dim(sample.table0)
sample.table0 <- sample.table0[sample.table0$sample %in% testID,] # 2011
table(sample.table0$cell,sample.table0$group1)
table(is.na(sample.table0$purity))
logcpm0 <- logcpm0[,sample.table0$sample]
count0 <- count0[,sample.table0$sample]

rna2 <- c("pri_miRNA","piRNA","rRNA","lncRNA","mRNA","snoRNA","snRNA","srpRNA","tRNA","tucpRNA","Y_RNA","ktRNA","Rfam")


sample.table0.bak <- sample.table0


 01A  11A 
2941  566 


TCGA-BLCA_small TCGA-BRCA_small TCGA-CHOL_small TCGA-COAD_small TCGA-ESCA_small 
            119             150              45             449             113 
TCGA-HNSC_small TCGA-KICH_small TCGA-KIRC_small TCGA-KIRP_small TCGA-LIHC_small 
            562              91             150             134             420 
TCGA-LUAD_small TCGA-LUSC_small TCGA-PRAD_small TCGA-STAD_small TCGA-THCA_small 
            555             145             150             141             150 
TCGA-UCEC_small 
            133 


              Stage I   Stage IA   Stage IB   Stage II  Stage IIA  Stage IIB 
       289        598        194        226        361        310        201 
 Stage IIC  Stage III Stage IIIA Stage IIIB Stage IIIC   Stage IV  Stage IVA 
         1        266        214        103         69        199        299 
 Stage IVB  Stage IVC    Stage X        T2a        T2b        T2c        T3a 
        20          6          1          4          1         75         41 
       T3b         T4 
        25          4 


            Stage I  Stage II Stage III Stage IVA Stage IVB Stage IVC 
       14        21        99       107       269        11         7 

           
            Normal Tumor
                 0    85
  -            566     0
  stage i        0   904
  stage ii       0   812
  stage iii      0   649
  stage iv       0   491

...

[1] 2068

[1] 3436   33

[1] 3436   33

      
       Metastasis Normal Primary
  BLCA         68     19      30
  BRCA         17     50      82
  CHOL          8      9      27
  COAD         40      8      60
  ESCA         34     13      65
  HNSC         75     44      23
  KICH         20     25      46
  KIRC         35     50      65
  KIRP         25     32      75
  LIHC         24     50      76
  LUAD         22     19      77
  LUSC         18     38      81
  PRAD         48     50      52
  STAD         36     36      56
  THCA         30     50      70
  UCEC         24     33      76


FALSE 
 2011 

In [20]:
# table(sample.table0$group)
# table(sample.table0$group1)
# table(sample.table0$group2)
sample.table0 <- sample.table0.bak
# sample.table0 <- sample.table0[sample.table0$group1 %in% c("Metastasis","Primary") & sample.table0$cell=="ESCA",]
dim(sample.table0)

[1] 2011   33

In [21]:
Metadata <- sample.table0 #[(sample.table0$Threatment=="-") & (sample.table0$treatmentnaive %in% c("-","0")),] # 1042
# Metadata <- sample.table0
dim(Metadata)
Metadata$purity <- 100*Metadata$purity # if skip this, affect 1st method !!!!!!!
# head(Metadata)

[1] 2011   33

In [22]:
# PCdata <- data.table::fread(paste0(diffpre,"/output/",dst,"/PCA/cpm.PCA.bycell.pdf.PCAdata.txt"),header = T,sep = "\t",data.table = F)
PCdata <- data.table::fread(paste0(diffpre,"/output/",dst,"/PCA/raw.cpm.PCA.bycell.png.PCAdata.txt"),header = T,sep = "\t",data.table = F)
# PCdata <- data.table::fread(paste0(diffpre,"/output/",dst,"/PCA/ESCA.cpm.PCA.bygroup1.tumor.png.PCAdata.txt"),header = T,sep = "\t",data.table = F)
rownames(PCdata) <- PCdata$sample
Metadata <- Metadata[Metadata$sample %in% rownames(PCdata),]

PCdata <- PCdata[Metadata$sample,]
dim(PCdata)

data_tmp=left_join(PCdata,Metadata,by="sample")
data_tmp[1:3,]



[1] 2011   51

,sample,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,⋯,lineage,ESTIMATE,ABSOLUTE,LUMP,IHC,CPE,stromal,immune,estimate,purity
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,TCGA-E2-A158-01A-11R-A12C-13_mirna_gdc_realn,1.7679,-4.6088,22.2677,-28.0268,10.178,1.0058,-0.048013,12.9109,-8.7881,⋯,Breast,0.9534,NA,NA,0.8,0.7799,-1137.1,-404.43,-1541.5,92.914
2,TCGA-BH-A0BJ-01A-11R-A057-13_mirna_gdc_realn,22.3835,-2.8802,4.1309,-19.0064,8.438,-7.3515,14.677781,9.9593,6.4464,⋯,Breast,0.7386,0.37,0.8453,0.7,0.7458,1023.0,764.21,1787.2,64.687
3,TCGA-BH-A208-01A-11R-A156-13_mirna_gdc_realn,-37.2530,-3.6125,15.0360,3.6979,17.407,2.1425,-5.747116,1.6722,16.1695,⋯,Breast,0.5951,0.31,0.6877,0.6,0.5642,2185.7,814.63,3000.3,50.156


## use variancePartion package (will drop NA)

In [28]:
# opt1: all N + T
varList2 <- list()
useCols <- c("cell","group","tumor_stage","gender","purity","age") # ,"race"
elbowPCA <- 16

for( i in 1:elbowPCA){
    # i<-1
    message(i)
    
    PCdata2=PCdata%>% #column_to_rownames("ID")%>%
        t() %>% as.data.frame()
    PCdata2=PCdata2[paste0("PC",i),] %>% mutate(across(where(is.character), as.numeric))

    # form = ~ (1|patient) + (1|Immunosuppressant) + PSLmg + (1|cell) + (1|group) + age + (1|sex)
    form = ~ (1|cell) + (1|group) + (1|tumor_stage) + (1|gender) + purity + age # + (1|race)
    

    varPart = fitExtractVarPartModel( PCdata2, form, Metadata ) %>% as.data.frame()
    #Samples with missing data will be dropped. !!!

    varPart <- varPart[,c(useCols,colnames(varPart)[!colnames(varPart) %in% useCols])]
    colnames(varPart) <- c(useCols,"residual")
    # colnames(varPart) <- c("Patient","Immunosuppressant","PSLmg","Cell","Group","Sex","Age","Residual")
    varList2[[i]] <- varPart
}

1

Warning message:
“Variables contain NA's: age 
Samples with missing data will be dropped.
”
Dividing work into 1 chunks...


Total:0.1 s

2

Warning message:
“Variables contain NA's: age 
Samples with missing data will be dropped.
”
Dividing work into 1 chunks...


Total:0.1 s

3

Warning message:
“Variables contain NA's: age 
Samples with missing data will be dropped.
”
Dividing work into 1 chunks...


Total:0.2 s

4

Warning message:
“Variables contain NA's: age 
Samples with missing data will be dropped.
”
Dividing work into 1 chunks...


Total:0.09 s

5

Warning message:
“Variables contain NA's: age 
Samples with missing data will be dropped.
”
Dividing work into 1 chunks...


Total:0.1 s

6

Warning message:
“Variables contain NA's: age 
Samples with missing data will be dropped.
”
Dividing work into 1 chunks...


Total:0.1 s

7

Warning message:
“Variables contain NA's: age 
Samples with missing data will be dropped.
”
Dividing work into 1 chunks...


Total:0.1 s

8

Warning m

In [45]:
# opt2: all tumor for given cancer
varList2 <- list()
useCols <- c("tumor_stage","gender","purity","age") # ,"race"
elbowPCA <- 8

for( i in 1:elbowPCA){
    # i<-1
    message(i)
    
    PCdata2=PCdata%>% #column_to_rownames("ID")%>%
        t() %>% as.data.frame()
    PCdata2=PCdata2[paste0("PC",i),] %>% mutate(across(where(is.character), as.numeric))

    # form = ~ (1|patient) + (1|Immunosuppressant) + PSLmg + (1|cell) + (1|group) + age + (1|sex)
    form = ~ (1|tumor_stage) + (1|gender) + purity + age # + (1|race)
    

    varPart = fitExtractVarPartModel( PCdata2, form, Metadata ) %>% as.data.frame()
    #Samples with missing data will be dropped. !!!

    varPart <- varPart[,c(useCols,colnames(varPart)[!colnames(varPart) %in% useCols])]
    colnames(varPart) <- c(useCols,"residual")
    # colnames(varPart) <- c("Patient","Immunosuppressant","PSLmg","Cell","Group","Sex","Age","Residual")
    varList2[[i]] <- varPart
}

1

Dividing work into 1 chunks...


Total:0.07 s

2

Dividing work into 1 chunks...


Total:0.03 s

3

Dividing work into 1 chunks...


Total:0.03 s

4

Dividing work into 1 chunks...


Total:0.03 s

5

Dividing work into 1 chunks...


Total:0.03 s

6

Dividing work into 1 chunks...


Total:0.03 s

7

Dividing work into 1 chunks...


Total:0.03 s

8

Dividing work into 1 chunks...


Total:0.03 s



In [29]:
varDf2 <- do.call(rbind,varList2) %>% t() %>% as.data.frame()
varDf2
sum(varDf2[,1])
#patientS200 patientS465 patientS524

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
cell,0.17890275,0.448934613,4.2445e-01,0.2886688,0.513149242,0.58702707,0.2887842,0.41762017,2.6388e-01,0.32273979,0.2537559,0.221565636,0.289440635,1.7381e-01,0.2722403,0.11597026297
group,0.12479601,0.035225743,0.0000e+00,0.0888825,0.084498214,0.04729566,0.0760492,0.06049865,8.7929e-10,0.02066178,0.0291055,0.002309152,0.077067226,3.8882e-02,0.0189864,0.00139097494
tumor_stage,0.01154966,0.002765223,9.0016e-03,0.0040929,0.000000000,0.00065875,0.0000000,0.00000000,9.0625e-03,0.03153643,0.0049271,0.007768070,0.002153539,2.3028e-03,0.0162735,0.00095251450
gender,0.00091740,0.004229246,6.1229e-11,0.0080569,0.001523683,0.00000000,0.0000000,0.00039055,0.0000e+00,0.00000000,0.0000000,0.000000000,0.000000000,1.3275e-10,0.0000000,0.00000000000
purity,0.03286689,0.001103597,7.2725e-03,0.0045711,0.002341545,0.00704628,0.0268900,0.02095110,8.0026e-02,0.00252452,0.0162519,0.014101002,0.000058399,1.2381e-02,0.0600279,0.00066869983
age,0.00052562,0.000028087,5.3004e-04,0.0012959,0.000052356,0.00048913,0.0022539,0.00043682,9.8903e-04,0.00023745,0.0041208,0.000060477,0.001324746,2.1651e-03,0.0011804,0.00000011751
residual,0.65044167,0.507713491,5.5874e-01,0.6044319,0.398434961,0.35748312,0.6060227,0.50010271,6.4604e-01,0.62230003,0.6918388,0.754195663,0.629955455,7.7045e-01,0.6312915,0.88101743024


[1] 1

In [30]:
pheatmap::pheatmap(mat = varDf2,
                    #    annotation_col = annotation_col, #data frame that specifies the annotations shown on left side of the heatmap.
                    #    annotation_row = annotation_row,
                    #    annotation_colors = ann_colors, # RColorBrewer::brewer.pal(6, "Set3")
                       border_color="NA",
                       display_numbers = round(varDf2,2),
                       scale = "none", # row,none
                       #labels_col = 3, labels_row = 6,
                       cluster_cols = F,cluster_rows = F,
                    #    gaps_row = sum(annotation_row$trend=="up"), # invalid if cluster row/col
                       #cutree_cols = 2,cutree_rows = 3,
                       show_colnames=T, show_rownames=T,
                       fontsize = 12,
                       height = 3, width = 6,
                       color = ( colorRampPalette( RColorBrewer::brewer.pal(n = 9, name = "OrRd"))(100) ),  # steelblue/dodgerblue4-firebrick2, viridis::viridis_pal()(100), #
                       #fontsize_row = 5,
                       filename = paste0("./figure/v2.varPar.cpm.heatmap3.pdf")
    )